# Solutions · 03 · Stability tests

Worked solutions to the exercises of [notebook 03](../notebooks/03_stability_tests.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, stability as st, polynomials as pl, linalg as la, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

The loop 1 + K (s + a)/(s^2 (s + 10)) = 0: use the Routh array symbolically (write the conditions on K and a) and confirm numerically for a = 1 and a = 5 the range of K for stability.

In [2]:
print("1 + K (s + a)/(s^2 (s + 10)) = 0  ->  s^3 + 10 s^2 + K s + K a = 0")
print("Routh: first column 1, 10, (10 K - K a)/10, K a  ->  stable iff K > 0, a > 0 and a < 10 (then K cancels from the third row's sign)")
for a in (1.0, 5.0, 12.0):
    verdicts = [st.routh([1.0, 10.0, K, K * a])["stable"] for K in (0.1, 1.0, 10.0, 100.0)]
    print(f"a = {a:4}: stable for K = 0.1, 1, 10, 100? {verdicts}")

1 + K (s + a)/(s^2 (s + 10)) = 0  ->  s^3 + 10 s^2 + K s + K a = 0
Routh: first column 1, 10, (10 K - K a)/10, K a  ->  stable iff K > 0, a > 0 and a < 10 (then K cancels from the third row's sign)
a =  1.0: stable for K = 0.1, 1, 10, 100? [True, True, True, True]
a =  5.0: stable for K = 0.1, 1, 10, 100? [True, True, True, True]
a = 12.0: stable for K = 0.1, 1, 10, 100? [False, False, False, False]


The array gives the condition symbolically: with the zero inside the double-integrator plant's left half-plane
(0 < a < 10) every positive gain stabilises the loop; with a > 10 none does. A root locus would show the same
(the zero must lie between the double pole at the origin and the pole at −10 for the branches to bend left).

## Exercise 2

For the levitated ball (`models.magnetic_levitation`) with proportional feedback u = -K x, show with the Routh array that no K stabilises it, and that u = -K x - Kd x' does for suitable Kd. Confirm with the eigenvalues.

In [3]:
ml = models.magnetic_levitation()["system"]
A, B = ml.A, ml.B
a, b = A[1, 0], B[1, 0]
print(f"x'' = {a:.1f} x + ({b:.2f}) u")
print(f"the input gain b is negative, so the stabilising sign is u = +K x: s^2 + 0 s + ({-b:.2f} K - {a:.1f}); the s coefficient is zero for every K, so the Routh array can never be all positive")
K = 100.0
print("   eigenvalues with u = K x (K = 100):", np.round(np.linalg.eigvals(A + B @ [[K, 0.0]]), 3), "(imaginary: marginal, never stable)")
Kd = 0.5
print(f"u = K x + Kd x' gives s^2 + ({-b:.2f} Kd) s + ({-b:.2f} K - {a:.1f}): positive coefficients for Kd > 0 and K > {a / -b:.2f}")
print("   eigenvalues with K = 100, Kd = 0.5:", np.round(np.linalg.eigvals(A + B @ [[K, Kd]]), 3))

x'' = 1962.0 x + (-28.01) u
the input gain b is negative, so the stabilising sign is u = +K x: s^2 + 0 s + (28.01 K - 1962.0); the s coefficient is zero for every K, so the Routh array can never be all positive
   eigenvalues with u = K x (K = 100): [0.+28.973j 0.-28.973j] (imaginary: marginal, never stable)
u = K x + Kd x' gives s^2 + (28.01 Kd) s + (28.01 K - 1962.0): positive coefficients for Kd > 0 and K > 70.04
   eigenvalues with K = 100, Kd = 0.5: [-7.004+28.114j -7.004-28.114j]


Position feedback alone cannot add damping to a system without any: the characteristic polynomial lacks its
s-term and the Routh array says marginal at best (and with the gain too small, two real poles, one unstable).
Velocity feedback (a derivative term, or a lead) supplies the damping; this is why every levitation and every
inverted-pendulum controller contains a derivative. The sign of the feedback follows the sign of the input
gain - a detail that reverses stability if overlooked.

## Exercise 3

**Implement it yourself:** solve the Lyapunov equation A^T P + P A = -Q for a 3 x 3 A by the Kronecker product (vec form) and compare with `linalg.lyapunov`; then show that P fails to be positive definite when A has an unstable eigenvalue.

In [4]:
rng = np.random.default_rng(0)
A = rng.normal(size=(3, 3)) - 3 * np.eye(3)
Q = np.eye(3)
K_ = np.kron(np.eye(3), A.T) + np.kron(A.T, np.eye(3))          # vec(A' P + P A) = (I x A' + A' x I) vec(P)
P_hand = np.linalg.solve(K_, -Q.reshape(-1, order="F")).reshape(3, 3, order="F")
print("Kronecker solution vs library:", f"{np.max(np.abs(P_hand - la.lyapunov(A.T, Q))):.1e}", "; eigenvalues of P:", np.round(np.linalg.eigvalsh(P_hand), 4))
A_un = A + 4 * np.eye(3)                                          # push an eigenvalue into the right half-plane
print("unstable A: eigenvalues", np.round(np.linalg.eigvals(A_un).real, 3), "-> P has eigenvalues", np.round(np.linalg.eigvalsh(la.lyapunov(A_un.T, Q)), 3), "(not all positive)")

Kronecker solution vs library: 0.0e+00 ; eigenvalues of P: [0.108  0.1497 0.2332]
unstable A: eigenvalues [ 1.79   0.635 -0.539] -> P has eigenvalues [-3.13  -0.277  6.44 ] (not all positive)


The vec trick turns the matrix equation into an ordinary linear system of n² unknowns; the solution is
symmetric and positive definite exactly when A is stable, and acquires a negative eigenvalue as soon as A
has an unstable one - Lyapunov's theorem in a computation.